0To train the model, the project utilizes a UTKFace dataset containing labeled facial images. The labels of each face image is embedded in the file name,
formated like [age]_[gender]_[race]_[date&time].jpg

[age] is an integer from 0 to 116, indicating the age

[gender] is either 0 (male) or 1 (female)

[race] is an integer from 0 to 4, denoting White=0, Black=1, Asian=2, Indian=3, and Others=4 (like Hispanic, Latino, Middle Eastern).

[date&time] is in the format of yyyymmddHHMMSSFFF, showing the date and time an image was collected to UTKFace


In [4]:
import os
import re
import random
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

import tensorflow.keras.applications
from PIL import Image
import pickle
import tensorflow as tf
from tensorflow.keras import Model, layers, regularizers
from utils.GradientReversal import GradientReversal
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import Huber

2025-08-07 11:58:09.893334: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Handle File names to extact age gender race

In [5]:
# add file folder path dir and using regular expression to
# parse the filenames

dataset_path = '../data/utkface_dataset'
pattern = r"(\d+)_(\d)_(\d)_(\d+).jpg"

image_size = (384, 384)
batch_size = 32

def parse_label(filename):
    match = re.match(pattern, filename)
    if match:
        age = int(match.group(1))
        gender = int(match.group(2))
        race = int(match.group(3))
        return age, gender, race
    else:
        raise IOError(f'{filename} Invalid filename')

def load_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, image_size)
    image = preprocess_input(image)

    age, gender, race = label
    return image, (age, gender, race)


def create_dataset(image_files):
    labels = [parse_label(os.path.basename(file)) for file in image_files]
    ages = tf.constant([label[0] for label in labels], dtype=tf.int32)
    genders = tf.constant([label[1] for label in labels], dtype=tf.int32)
    races = tf.constant([label[2] for label in labels], dtype=tf.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_files, (ages, genders, races)))
    dataset = dataset.map(lambda x, y: (load_image(x, y)), num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


Raw dataset preparation

In [ ]:
def plot_distribution(files):
    ages = [parse_label(os.path.basename(f))[0] for f in files]
    genders = [parse_label(os.path.basename(f))[1] for f in files]
    races = [parse_label(os.path.basename(f))[2] for f in files]

    plt.figure(figsize=(8, 5))
    plt.hist(ages, bins=24, color='yellow', edgecolor='black')
    plt.title('Age Distribution in UTKFace Dataset')
    plt.xlabel('Age')
    plt.ylabel('Frenquency')
    plt.show()

    plt.figure(figsize=(8, 5))
    gender_count = Counter(genders)
    plt.bar(gender_count.keys(), gender_count.values(), color=['yellow', 'pink'])
    plt.xticks([0,1], ['Male', 'Female'])
    plt.title('Gender Distribution in UTKFace Dataset')
    plt.xlabel('Gender')
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(8, 5))
    race_counts = Counter(races)
    # White, Black, Asian, Indian, and Others (like Hispanic, Latino, Middle Eastern).
    plt.bar(race_counts.keys(), race_counts.values(), color=['gray','black', 'yellow', 'brown', 'red'])
    plt.xticks([0,1,2,3,4],['White-0', 'Black-1', 'Asian-2', 'Indian-3', 'Others-4'])
    plt.title('Race Distribution in UTKFace Dataset')
    plt.xlabel('Race')
    plt.ylabel('Count')
    plt.show()

all_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith('.jpg')]

plot_distribution(all_files)


In [ ]:
def stratified_downsampling(files, age_bins, target_per_bin=1000):
    random.seed(42)
    print("the value of target_per_bin ", target_per_bin)

    race_white = 0
    age_to_files = {i: [] for i in range(len(age_bins) - 1)}

    for f in files:
        age, _, race = parse_label(os.path.basename(f))
        for i in range(len(age_bins)-1):
            if age_bins[i] <= age < age_bins[i+1]:
                age_to_files[i].append((f, race))
                break

    selected_files = []
    for age_bin_idx, age_bin_files in age_to_files.items():
        if len(age_bin_files) > target_per_bin:
            white_files = [f for f, race in age_bin_files if race==race_white]

            non_white_files = [f for f, race in age_bin_files if race != race_white]

            if len(non_white_files)> target_per_bin:
                 selected_files.extend(random.sample(non_white_files, target_per_bin))
            else:
                selected_files.extend(non_white_files)
                remaining_needed = target_per_bin - len(non_white_files)
                selected_files.extend(random.sample(white_files, remaining_needed))
        else:
            selected_files.extend([f for f, _ in age_bin_files])
    return selected_files

age_bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100]

downsampled_files = stratified_downsampling(all_files, age_bins, target_per_bin=1000)

print("the length of the downsampled_files ", len(downsampled_files))

plot_distribution(downsampled_files)

In [ ]:
# Get all image files
total_files = len(downsampled_files)

# Shuffle files with a consistent seed
seed = 42
random.seed(seed)
random.shuffle(downsampled_files)

# Split the data into train, validation, and test
train_split = 0.5
val_split = 0.25
test_split = 0.25


train_count = int(total_files * train_split)
val_count = int(total_files * val_split)
test_count = total_files - train_count - val_count

train_files = downsampled_files[:train_count]
val_files = downsampled_files[train_count:train_count + val_count]
test_files = downsampled_files[train_count + val_count:]

# Create dataset for each subset
train_ds = create_dataset(train_files)
val_ds = create_dataset(val_files)
test_ds = create_dataset(test_files)

In [ ]:
def show_image_samples(dataset, subset_name, num_samples=5):
    plt.figure(figsize=(20, 6))
    gender_map = {0: 'Male', 1: 'Female'}
    race_map = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Indian', 4: 'Others'}

    for images, (ages, genders, races) in dataset.take(1):
        for j in range(num_samples):
            ax = plt.subplot(1, num_samples, j + 1)
            plt.imshow(images[j].numpy())
            age = ages[j].numpy()
            gender = genders[j].numpy()
            race = races[j].numpy()
            plt.title(f"Age: {age}\n Gender: {gender_map[gender]} \n Race: {race_map[race]}", fontsize=100/num_samples)
            plt.axis("off")

    plt.suptitle(f"{subset_name} Samples", fontsize=20)
    plt.show()
    print('\n')

# Display some image samples and labels from each subset
show_image_samples(train_ds, "Training",)
show_image_samples(val_ds, "Validation")
show_image_samples(test_ds, "Testing")

In [ ]:
def plot_age_distribution(dataset, subset_name):
    ages = []
    for _, (age_labels, _, _) in dataset:
        ages.extend(age_labels.numpy())

    plt.figure(figsize=(6, 4))
    plt.hist(ages, bins=range(0, 120, 5), edgecolor='black')
    plt.title(f"Age Distribution in {subset_name} Set")
    plt.xlabel("Age")
    plt.ylabel("Frequency")
    plt.show()

# Plot age distributions for training, validation, and test sets
plot_age_distribution(train_ds, "Training")
plot_age_distribution(val_ds, "Validation")
plot_age_distribution(test_ds, "Testing")

In [ ]:
def plot_gender_distribution(dataset, subset_name):
    gender_count = {0: 0, 1: 0}
    for _, (_, gender_labels, _) in dataset:
        unique, counts = np.unique(gender_labels.numpy(), return_counts=True)
        for u, c in zip(unique, counts):
            gender_count[u] += c

    plt.figure(figsize=(6, 4))
    plt.bar(['Male', 'Female'], [gender_count[0], gender_count[1]], color=['blue', 'pink'])
    plt.title(f"Gender Distribution in {subset_name} Set")
    plt.xlabel("Gender")
    plt.ylabel("Count")
    plt.show()

# Plot gender distributions for training, validation, and test sets
plot_gender_distribution(train_ds, "Training")
plot_gender_distribution(val_ds, "Validation")
plot_gender_distribution(test_ds, "Testing")

In [ ]:
def plot_race_distribution(dataset, subset_name):
    race_count = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
    for _, (_, _, race_labels) in dataset:
        unique, counts = np.unique(race_labels.numpy(), return_counts=True)
        for u, c in zip(unique, counts):
            race_count[u] += c

    plt.figure(figsize=(6, 4))
    plt.bar(['White', 'Black', 'Asian', 'Indian', 'Others'],
            [race_count[i] for i in range(5)], color=['gray', 'black', 'yellow', 'brown', 'green'])
    plt.title(f"Race Distribution in {subset_name} Set")
    plt.xlabel("Race")
    plt.ylabel("Count")
    plt.show()

# Plot race distributions for training, validation, and test sets
plot_race_distribution(train_ds, "Training")
plot_race_distribution(val_ds, "Validation")
plot_race_distribution(test_ds, "Testing")

In [6]:
import os
import certifi

os.environ['SSL_CERT_FILE'] = certifi.where()

In [7]:
input_shape = (384, 384, 3)

base_model = tf.keras.applications.EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape
)

base_model.summary()

Model: "efficientnetv2-s"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 384, 384,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 384, 384,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 192, 192,  │        648 │ rescaling[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 192, 192,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 192, 192,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 192, 192,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 192, 192,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 192, 192,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 192, 192,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 192, 192,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 192, 192,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 192, 192,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 192, 192,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 192, 192,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 96, 96,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 96, 96,    │        384 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 96, 96,    │          0 │ block2a_expand_b

 Total params: 20,331,360 (77.56 MB)

 Trainable params: 20,177,488 (76.97 MB)

 Non-trainable params: 153,872 (601.06 KB)

In [ ]:
base_model.trainable = True
# freeze all the layer in efficientnetv2s, except for the 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

#freeze all layers and adding age, gender, race output
inputs = tf.keras.layers.Input(shape=input_shape)

x = base_model(inputs, training=True)
x = layers.GlobalAveragePooling2D()(x)

shared_features = layers.Dropout(0.3)(x)

dense_age = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01))(shared_features)
bn_age = layers.BatchNormalization()(dense_age)
dropout_age = layers.Dropout(0.3)(bn_age)

dense_age_1 = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))(dropout_age)
bn_age_1 = layers.BatchNormalization()(dense_age_1)
dropout_age_1 = layers.Dropout(0.3)(bn_age_1)

age_output = layers.Dense(1, activation='relu',name='age')(dropout_age_1)

grl_gender = GradientReversal(max_lambda=1.0)(shared_features)

dense_gender = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))(grl_gender)
bn_gender = layers.BatchNormalization()(dense_gender)
dropout_gender = layers.Dropout(0.3)(bn_gender)
gender_output = layers.Dense(1, activation='sigmoid', name='gender')(dropout_gender)

grl_race = GradientReversal(max_lambda=1.0)(shared_features)

dense_race = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))(grl_race)
bn_race = layers.BatchNormalization()(dense_race)
dropout_race = layers.Dropout(0.3)(bn_race)
race_output = layers.Dense(5, activation='softmax', name='race')(dropout_race)

model = Model(inputs=inputs, outputs=[age_output, gender_output, race_output])

In [ ]:
huber_loss = Huber()
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'age': huber_loss,
        'gender': 'binary_crossentropy',
        'race': 'sparse_categorical_crossentropy',
    },
    loss_weights={
        'age': 1.5,
        'gender': 0.5,
        'race': 0.5,
    },
    metrics={
        'age': 'mae',
        'gender': 'accuracy',
        'race': 'accuracy',
    }
)

In [ ]:
model.summary()

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_age_mae',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_age_mae',
    patience=15,
    mode='min',
    restore_best_weights=True,
    verbose=1
)

In [ ]:
from utils.GradientReversal import GradientReversal

epochs = 50
lambda_start = 0.0
lambda_max = 1.0

# 获取 GradientReversal 层引用
grl_gender_layer = [layer for layer in model.layers if isinstance(layer, GradientReversal)][0]
grl_race_layer = [layer for layer in model.layers if isinstance(layer, GradientReversal)][1]

for epoch in range(epochs):
    # 线性增长 lambda
    current_lambda = lambda_start + (lambda_max - lambda_start) * epoch / (epochs - 1)
    grl_gender_layer.set_lambda(current_lambda)
    grl_race_layer.set_lambda(current_lambda)

    print(f"\nEpoch {epoch + 1}/{epochs} - λ = {current_lambda:.4f}")

    # 每次训练一轮
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=1,
        verbose=1,
        callbacks=[reduce_lr, early_stopping]
    )


Model Evaluation

In [ ]:
# Plot evaluation metrics for age, gender, and race
plt.figure(figsize=(8, 4))
plt.plot(history.history['age_mae'], label='Train Age MAE')
plt.plot(history.history['val_age_mae'], label='Validation Age MAE')
plt.title('Age Mean Absolute Error')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['gender_accuracy'], label='Train Gender Accuracy')
plt.plot(history.history['val_gender_accuracy'], label='Validation Gender Accuracy')
plt.title('Gender Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['race_accuracy'], label='Train Race Accuracy')
plt.plot(history.history['val_race_accuracy'], label='Validation Race Accuracy')
plt.title('Race Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate on the test dataset
test_loss, test_age_loss, test_gender_loss, test_race_loss, test_age_mae, test_gender_accuracy, test_race_accuracy = model.evaluate(test_ds)
print('Test Gender Accuracy:', test_gender_accuracy)
print('Test Race Accuracy:', test_race_accuracy)
print('Test Age MAE:', test_age_mae)


In [ ]:
# Save model efficientnet h5 and history pickle
with open('../models/history.pkl', 'wb') as file:
    pickle.dump(history.history, file)
model.save('../models/efficientnet_v2s_dev_reduce.h5')
model.save('../models/efficientnet_v2s_dev_reduce')

In [ ]:
# Show preview image with true label and prediction from data test
for images, labels in test_ds.take(1):
    batch_size = images.shape[0]
    num_images = 5
    selected_indices = random.sample(range(batch_size), num_images)
    selected_images = tf.gather(images, selected_indices)

    ages = tf.gather(labels[0], selected_indices)
    genders = tf.gather(labels[1], selected_indices)
    races = tf.gather(labels[2], selected_indices)

    predictions = model.predict(selected_images)
    pred_ages = predictions[0].flatten()
    pred_genders = predictions[1].flatten()
    pred_races = predictions[2].argmax(axis=1)
    pred_race_confidences = predictions[2].max(axis=1)

    plt.figure(figsize=(15, 15))
    for i in range(num_images):
        true_age = ages[i].numpy()
        true_gender = 'Male' if genders[i].numpy() == 0 else 'Female'
        true_race = ['White', 'Black', 'Asian', 'Indian', 'Others'][races[i].numpy()]

        pred_age = int(pred_ages[i])
        pred_gender = 'Male' if pred_genders[i] < 0.5 else 'Female'
        gender_confidence = 1 - pred_genders[i] if pred_genders[i] < 0.5 else pred_genders[i]
        pred_race = ['White', 'Black', 'Asian', 'Indian', 'Others'][pred_races[i]]
        race_confidence = pred_race_confidences[i]

        plt.subplot(1, num_images, i + 1)

        plt.imshow(selected_images[i].numpy().astype("uint8"))
        plt.title(f"Age: {true_age}\nGender: {true_gender}\nRace: {true_race}")
        plt.axis('off')

        plt.text(0.5, -0.1,
                 f"Age Pred: {pred_age}\nGender Pred: {pred_gender} ({gender_confidence:.2f})\nRace Pred: {pred_race} ({race_confidence:.2f})",
                 fontsize=12, ha='center', va='top', transform=plt.gca().transAxes)

    plt.show()